# 05. 更新対象の固定予算比較（GPU 常駐 batch）

Fashion-MNIST の TT-MLP について、同じ Stage C 初期状態と同じ 3,200 full-batch 更新予算で、更新対象だけを変えて比べる。収束後の最適性能の比較ではない。

比較する scope は次の3つ。

- `joint`: 結合した TT pair と通常 parameter を更新する。学習率はどちらも 0.3。
- `tt_only`: 結合した TT pair だけを更新する。通常 parameter は勾配保存、更新候補、書き戻しから外す。forward は残す。
- `dense_only`: 通常 parameter だけを更新する。TT の結合、局所勾配更新、SVD、rank 制限、core 書き戻し、正準中心の移動、pair 巡回は行わない。

3方式とも、batch size 4,096 と、前処理済み学習データの GPU 事前配置を使う。これは Notebook 04 で採ることにした供給方法であり、04 の短時間ベンチマーク自体を合否には使わない。

無更新 baseline は、この共通初期 checkpoint の評価である。3,200 更新の「更新なし」条件は走らせない。

test と rank-selection validation の中身は走査しない。学習は train、評価と best 選択は early-stopping validation だけを使う。early stopping はしない。


## 1. 設定

このセルは定数、保存先、実行フラグを置く。CUDA は初期化しない。

- `RUN_SMOKE`: 旧 joint と新 joint の 80 更新比較、および3 scope の 80 更新検証。既定は `False`。
- `RUN_MAIN_EXPERIMENT`: 3,200 更新の3条件。既定は `False`。保存済み smoke が一致したときだけ進む。
- `SAVE_RESULTS`: 実行したときだけ 05 専用のディレクトリへ書く。

`None` の学習率は、その側を更新処理から除外するという意味である。学習率 0 で固定は表さない。

このセルの値だけでは、GPU 常駐が数値的に妥当だとは保証しない。それは smoke の検証対象である。


In [33]:
import hashlib
import json
import math
import os
import subprocess
import sys
import time
from pathlib import Path

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import numpy as np
import torch
from torch.utils.data import DataLoader

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    src_dir = candidate / "src"
    if (src_dir / "nn_compression").is_dir():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break
else:
    raise RuntimeError("src/nn_compression が見つかりません。")

from nn_compression.compression import TTLinear
from nn_compression.datasets import get_fashion_mnist_datasets, split_fashion_mnist_dataset
from nn_compression.models import MNISTMLP
from nn_compression.training import (
    GpuResidentTensorBatches,
    TTUpdateScope,
    assert_scope_budget_completed,
    incomplete_scope_names,
    load_condition_summary,
    reject_incomplete_scope_rows,
    require_recorded_smoke_gate,
    run_resumable_condition_grid,
    save_scoped_full_batch_run,
    save_tt_linear_full_batch_sweep_run,
    train_scoped_full_batch_budget,
    train_tt_linear_full_batch_sweeps,
)
from nn_compression.tt_mps import (
    tt_matrix_canonicality_errors,
    tt_matrix_ranks,
    tt_matrix_to_dense,
)
from nn_compression.utils import find_project_root, get_experiment_dirs

# 長い学習は、ユーザーがフラグを True にしたときだけ始める。
RUN_SMOKE = True
RUN_MAIN_EXPERIMENT = True
SAVE_RESULTS = True

SEED = 0
BATCH_SIZE = 4096
DATA_RESIDENCY = "gpu_resident"
TOTAL_FULL_BATCH_UPDATES = 3200
EVALUATION_INTERVAL = 80
JOINT_TT_LEARNING_RATE = 0.3
JOINT_DENSE_LEARNING_RATE = 0.3
TT_ONLY_TT_LEARNING_RATE = 0.3
TT_ONLY_DENSE_LEARNING_RATE = None
DENSE_ONLY_TT_LEARNING_RATE = None
DENSE_ONLY_DENSE_LEARNING_RATE = 0.3
MAX_SWEEPS = 40
LOCAL_STEPS_PER_PAIR = 8
RANK_CAP = 12
EARLY_STOPPING_ENABLED = False
SMOKE_UPDATES = 80
SMOKE_OLD_BATCH_SIZE = 512
DIAGNOSTIC_BATCH_SIZE = 512
VALIDATION_BATCH_SIZE = 512
OUT_MODES = (4, 2, 2, 2, 4, 4)
IN_MODES = (2, 2, 2, 2, 7, 7)
EXPECTED_INITIAL_RANKS = (1, 8, 12, 12, 12, 12, 1)
LAYER_NAME = "fc1"
N_CORES = 6
N_TRAIN = 50_000
SPLIT_LENGTHS = (50_000, 5_000, 5_000)
EXPECTED_UPDATED_PARAMETERS = (
    "fc1.bias",
    "fc2.weight",
    "fc2.bias",
    "fc3.weight",
    "fc3.bias",
)
SMOKE_VALIDATION_LOSS_ABSOLUTE_TOLERANCE = 1e-3
SMOKE_VALIDATION_ACCURACY_ABSOLUTE_TOLERANCE = 0.001
EXPERIMENT_NAME = "05_update_scope_ablation_gpu_resident"
SMOKE_EXPERIMENT_NAME = "05_update_scope_ablation_gpu_resident_smoke"
NOTEBOOK_03_REFERENCE = {
    "batch_size": 512,
    "data_source": "dataloader",
    "joint_learning_rate": 0.3,
    "full_batch_updates": 3200,
    "validation_loss": 0.3426136145591736,
    "validation_accuracy": 0.8710,
}
PROTECTED_NAMES = {
    "00_full_batch_two_core_sweep_training",
    "01_tt_initialization_scale_diagnostics",
    "02_full_batch_sweep_update_budget_grid",
    "02_full_batch_sweep_update_budget_grid_smoke",
    "02_full_batch_sweep_update_budget_grid_smoke_bs512",
    "03_joint_learning_rate_02_03_s40_l8",
    "03_joint_learning_rate_02_03_s40_l8_smoke",
    "04_full_batch_gradient_optimization_benchmark",
}
CHECK_INDICES = (0, 511, 512, 4095, 4096, 49151, 49152, 49663, 49664, 49999)
SETTING_FIELDS = (
    "scope",
    "batch_size",
    "data_residency",
    "tt_learning_rate",
    "dense_learning_rate",
    "total_updates",
    "evaluation_interval",
    "local_steps_per_pair",
    "max_sweeps",
    "seed",
    "rank_cap",
    "checkpoint_sha256",
    "code_sha256",
    "split_index_sha256",
    "image_sha256",
    "label_sha256",
)
SUMMARY_FIELDS = (
    "condition_id",
    *SETTING_FIELDS,
    "stop_reason",
    "completed_updates",
    "best_update_count",
    "best_validation_loss",
    "best_validation_accuracy",
    "final_validation_loss",
    "final_validation_accuracy",
    "initial_validation_loss",
    "initial_validation_accuracy",
    "preload_seconds",
    "preload_peak_allocated_bytes",
    "preload_peak_reserved_bytes",
    "seconds_train",
    "seconds_eval",
    "seconds_save",
    "seconds_total",
    "peak_allocated_bytes",
    "peak_reserved_bytes",
    "svd_call_count",
    "dense_candidate_count",
    "core_writeback_count",
)
SMOKE_GATE_FIELDS = (
    "checkpoint_sha256",
    "image_sha256",
    "label_sha256",
    "split_index_sha256",
    "code_sha256",
    "seed",
    "rank_cap",
    "smoke_updates",
    "evaluation_interval",
    "joint_tt_learning_rate",
    "joint_dense_learning_rate",
    "old_batch_size",
    "old_data_residency",
    "new_batch_size",
    "new_data_residency",
    "old_new_joint_passed",
    "joint_scope_passed",
    "tt_only_scope_passed",
    "dense_only_scope_passed",
    "scope_checks_passed",
)

assert not EARLY_STOPPING_ENABLED
assert TOTAL_FULL_BATCH_UPDATES == MAX_SWEEPS * (2 * (N_CORES - 1) * LOCAL_STEPS_PER_PAIR)
assert EVALUATION_INTERVAL == 2 * (N_CORES - 1) * LOCAL_STEPS_PER_PAIR
assert SMOKE_UPDATES == EVALUATION_INTERVAL
assert TT_ONLY_DENSE_LEARNING_RATE is None
assert DENSE_ONLY_TT_LEARNING_RATE is None
assert SMOKE_VALIDATION_ACCURACY_ABSOLUTE_TOLERANCE == 0.001
assert DATA_RESIDENCY == "gpu_resident"
assert BATCH_SIZE == 4096
assert EXPERIMENT_NAME not in PROTECTED_NAMES
assert SMOKE_EXPERIMENT_NAME not in PROTECTED_NAMES

project_root = find_project_root(Path.cwd())
INITIAL_CHECKPOINT = (
    project_root
    / "models"
    / "40_dmrg"
    / "10_fashion_mnist_mlp"
    / "01_tt_initialization_scale_diagnostics"
    / "initial_stage_c.pt"
)
SCOPED_MODULE = project_root / "src" / "nn_compression" / "training" / "tt_linear_scoped_update.py"
_, models_dir, results_dir = get_experiment_dirs(
    project_root, "40_dmrg", "10_fashion_mnist_mlp", EXPERIMENT_NAME
)
_, smoke_models_dir, smoke_results_dir = get_experiment_dirs(
    project_root, "40_dmrg", "10_fashion_mnist_mlp", SMOKE_EXPERIMENT_NAME
)


def assert_output_path(path):
    resolved = Path(path).resolve()
    allowed = {
        models_dir.resolve(),
        results_dir.resolve(),
        smoke_models_dir.resolve(),
        smoke_results_dir.resolve(),
    }
    if not any(resolved == root or root in resolved.parents for root in allowed):
        raise RuntimeError(f"05 の保存先の外です: {resolved}")
    text = resolved.as_posix()
    for name in PROTECTED_NAMES:
        if name in text:
            raise RuntimeError(f"00〜04 の保存先へ書き込もうとしました: {resolved}")


for directory in (models_dir, results_dir, smoke_models_dir, smoke_results_dir):
    assert_output_path(directory)
    if directory.name in PROTECTED_NAMES or not directory.name.startswith("05_"):
        raise RuntimeError(f"保存先の名前が 05 専用ではありません: {directory}")

print("RUN_SMOKE:", RUN_SMOKE)
print("RUN_MAIN_EXPERIMENT:", RUN_MAIN_EXPERIMENT)
print("SAVE_RESULTS:", SAVE_RESULTS)
print("experiment:", EXPERIMENT_NAME)
print("smoke:", SMOKE_EXPERIMENT_NAME)
print("このセルは CUDA を初期化しません。")


RUN_SMOKE: True
RUN_MAIN_EXPERIMENT: True
SAVE_RESULTS: True
experiment: 05_update_scope_ablation_gpu_resident
smoke: 05_update_scope_ablation_gpu_resident_smoke
このセルは CUDA を初期化しません。


## 2. データ、checksum、Stage C

train 50,000、early-stopping validation 5,000、rank-selection validation 5,000 に分割する。test は長さだけを見て、中身は走査しない。rank-selection も index の識別子だけを作り、画像は読まない。

学習画像とラベルは sample 順に CPU へ積む。checksum、境界 sample、batch 512 と 4,096 の端数 batch をここで確認する。GPU への配置は smoke または主実験を開始したときだけ行う。

Stage C は `initial_stage_c.pt` を読む。checkpoint の path、SHA-256、state の key、parameter、buffer、TT core、tensorization、rank 上限、初期 rank 列、正準中心、seed、dtype を照合する。device は、実際に学習を始める瞬間に選んだ GPU と照合する。

この照合は、3方式が同じ初期状態から始まるための確認である。学習後の優劣は保証しない。


In [34]:
def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def tensor_sha256(tensor):
    array = tensor.detach().contiguous().cpu().numpy()
    return hashlib.sha256(array.tobytes()).hexdigest()


def index_sha256(subset):
    indices = torch.tensor(list(subset.indices), dtype=torch.int64)
    return hashlib.sha256(indices.numpy().tobytes()).hexdigest()


def make_loader(dataset, batch_size):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=0,
        pin_memory=False,
    )


# CUDA を初期化しない乱数設定。set_seed は学習開始時だけ呼ぶ。
np.random.seed(SEED)
torch.manual_seed(SEED)

train_full, test_dataset = get_fashion_mnist_datasets(
    project_root / "data",
    download=False,
)
train_dataset, early_stopping_dataset, rank_selection_dataset = split_fashion_mnist_dataset(
    train_full,
    SPLIT_LENGTHS,
    seed=SEED,
)
if (len(train_dataset), len(early_stopping_dataset), len(rank_selection_dataset)) != SPLIT_LENGTHS:
    raise RuntimeError("分割サイズが 50,000 / 5,000 / 5,000 ではありません。")
if len(test_dataset) != 10_000:
    raise RuntimeError("test の長さが 10,000 ではありません。中身は読みません。")

train_images = torch.stack([train_dataset[index][0] for index in range(len(train_dataset))])
train_labels = torch.tensor(
    [int(train_dataset[index][1]) for index in range(len(train_dataset))],
    dtype=torch.int64,
)
if train_images.shape[0] != N_TRAIN or train_labels.shape[0] != N_TRAIN:
    raise RuntimeError("学習 sample 数が 50,000 ではありません。")
IMAGE_SHA256 = tensor_sha256(train_images)
LABEL_SHA256 = tensor_sha256(train_labels)
for index in CHECK_INDICES:
    image, label = train_dataset[index]
    if not torch.equal(train_images[index], image) or int(train_labels[index]) != int(label):
        raise RuntimeError(f"境界 sample {index} が subset と一致しません。")

for batch_size, expected_last in ((512, 336), (4096, 848)):
    loader = make_loader(train_dataset, batch_size)
    sizes = []
    seen = 0
    for images, labels in loader:
        sizes.append(int(images.shape[0]))
        seen += int(images.shape[0])
        if images.shape[0] != labels.shape[0]:
            raise RuntimeError("画像とラベルの batch 次元が一致しません。")
    if seen != N_TRAIN or sizes[-1] != expected_last:
        raise RuntimeError(f"batch {batch_size} の走査が全 sample を過不足なく覆っていません: {sizes[-1]}")
    resident = GpuResidentTensorBatches(train_images, train_labels)
    resident_sizes = [int(images.shape[0]) for images, _ in resident.batches(batch_size)]
    if resident_sizes != sizes:
        raise RuntimeError("GPU 常駐用の slice と DataLoader の batch サイズ列が一致しません。")
    again = [int(images.shape[0]) for images, _ in resident.batches(batch_size)]
    if again != resident_sizes:
        raise RuntimeError("常駐 batch を再走査できません。")

loaded = torch.cat([images for images, _ in make_loader(train_dataset, BATCH_SIZE)])
if not torch.equal(loaded, train_images):
    raise RuntimeError("DataLoader の sample 順が積み上げた学習画像と一致しません。")

SPLIT_INDEX_SHA256 = "|".join(
    index_sha256(subset)
    for subset in (train_dataset, early_stopping_dataset, rank_selection_dataset)
)
CODE_SHA256 = file_sha256(SCOPED_MODULE)
SHARED_CHECKPOINT_SHA256 = file_sha256(INITIAL_CHECKPOINT)
try:
    GIT_HEAD = subprocess.check_output(
        ["git", "rev-parse", "HEAD"],
        cwd=project_root,
        text=True,
    ).strip()
except (OSError, subprocess.CalledProcessError):
    GIT_HEAD = "unavailable"


def core_keys(state):
    prefix = f"{LAYER_NAME}.cores."
    return sorted(
        (key for key in state if key.startswith(prefix)),
        key=lambda key: int(key.rsplit(".", 1)[-1]),
    )


def assert_checkpoint_contract(payload):
    if payload.get("layer_name") != LAYER_NAME:
        raise ValueError(f"layer_name が一致しません: {payload.get('layer_name')!r}")
    if tuple(payload.get("out_modes", ())) != OUT_MODES:
        raise ValueError(f"out_modes が一致しません: {payload.get('out_modes')!r}")
    if tuple(payload.get("in_modes", ())) != IN_MODES:
        raise ValueError(f"in_modes が一致しません: {payload.get('in_modes')!r}")
    if int(payload.get("rank_cap", -1)) != RANK_CAP:
        raise ValueError(f"rank_cap が一致しません: {payload.get('rank_cap')!r}")
    if int(payload.get("canonical_center", -1)) != 0:
        raise ValueError(f"canonical_center は 0 である必要があります: {payload.get('canonical_center')!r}")
    if payload.get("stage") != "C":
        raise ValueError(f"stage C ではありません: {payload.get('stage')!r}")
    if int(payload.get("seed", -1)) != SEED:
        raise ValueError(f"seed が一致しません: {payload.get('seed')!r}")


def build_model_from_checkpoint():
    """Stage C から CPU 上のモデルを作る。GPU へは学習開始時に移す。"""
    payload = torch.load(INITIAL_CHECKPOINT, map_location="cpu", weights_only=False)
    assert_checkpoint_contract(payload)
    state = payload["state_dict"]
    keys = core_keys(state)
    if len(keys) != len(OUT_MODES):
        raise ValueError(f"core 数が一致しません: {len(keys)}")
    cores = [state[key].detach().to(dtype=torch.float32, device="cpu") for key in keys]
    ranks = tt_matrix_ranks(cores)
    if ranks != EXPECTED_INITIAL_RANKS:
        raise ValueError(f"初期 rank 列が一致しません: {ranks}")
    errors = tt_matrix_canonicality_errors(cores, OUT_MODES, IN_MODES, 0)
    if errors["left"]:
        raise ValueError("site 0 正準のはずが、左側の直交誤差があります。")
    ortho_error = max(float(item) for item in errors["right"])
    if ortho_error > 1e-4:
        raise ValueError(f"site 0 の右直交誤差が大きすぎます: {ortho_error}")
    model = MNISTMLP()
    model.fc1 = TTLinear(cores, OUT_MODES, IN_MODES, bias=state[f"{LAYER_NAME}.bias"].detach())
    own = dict(model.named_parameters())
    if set(own) != set(state):
        raise ValueError(
            "checkpoint とモデルの parameter 名が一致しません: "
            f"extra={sorted(set(own) - set(state))} missing={sorted(set(state) - set(own))}"
        )
    dense_names = tuple(name for name in own if not name.startswith(f"{LAYER_NAME}.cores."))
    if dense_names != EXPECTED_UPDATED_PARAMETERS:
        raise ValueError(f"通常 parameter が想定と違います: {dense_names}")
    if list(model.named_buffers()):
        raise ValueError(f"buffer がある状態は想定外です: {list(model.named_buffers())}")
    with torch.no_grad():
        for key, value in state.items():
            if key.startswith(f"{LAYER_NAME}.cores.") or key == f"{LAYER_NAME}.bias":
                continue
            own[key].copy_(value)
    loaded_state = model.state_dict()
    for key, value in state.items():
        current = loaded_state[key]
        if current.dtype != torch.float32 or current.device.type != "cpu":
            raise TypeError(f"{key} が CPU float32 ではありません: {current.dtype} {current.device}")
        if not torch.equal(current, value.detach().to(dtype=torch.float32, device="cpu")):
            raise ValueError(f"初期 state が checkpoint と一致しません: {key}")
    for core in model.fc1.cores:
        core.requires_grad_(False)
    return model, float(ortho_error)


probe_model, initial_ortho_error = build_model_from_checkpoint()
if file_sha256(INITIAL_CHECKPOINT) != SHARED_CHECKPOINT_SHA256:
    raise RuntimeError("checkpoint を読んだだけで共通ファイルの内容が変わりました。")
diagnostic_images = train_images[:DIAGNOSTIC_BATCH_SIZE].contiguous()
diagnostic_labels = train_labels[:DIAGNOSTIC_BATCH_SIZE].contiguous()
train_eval_loader = make_loader(train_dataset, VALIDATION_BATCH_SIZE)
validation_loader = make_loader(early_stopping_dataset, VALIDATION_BATCH_SIZE)
del probe_model

print("checkpoint:", INITIAL_CHECKPOINT)
print("checkpoint sha256:", SHARED_CHECKPOINT_SHA256)
print("image sha256:", IMAGE_SHA256)
print("label sha256:", LABEL_SHA256)
print("split index sha256:", SPLIT_INDEX_SHA256)
print("code sha256:", CODE_SHA256)
print("git head:", GIT_HEAD)
print("initial ranks:", EXPECTED_INITIAL_RANKS)
print("site-0 right orthogonality max error:", initial_ortho_error)
print("test length, not scanned:", len(test_dataset))
print("rank-selection length, images not scanned:", len(rank_selection_dataset))
print("このセルは CUDA を初期化しません。")


checkpoint: D:\dev\nn-compression-svd-dmrg\models\40_dmrg\10_fashion_mnist_mlp\01_tt_initialization_scale_diagnostics\initial_stage_c.pt
checkpoint sha256: ef9d5afcbffc0cdad787bc7e17f230917ae7cfb780bd506f5a348ae38ce10c6a
image sha256: 6a1fd0602db5a1066210a55375d18f9b12115269e7c6759f64504735f020d4d5
label sha256: 6ead6f3f16028865d36fd3c8625eba751820bd8031265a3be03e206136a3c96c
split index sha256: aaed0a23690e2e3c0fd3b21c21ec28adb2606b17941b98087503382c3372f913|12cf7fe6cc2cdbb43177107c8a30dffb86cc099386d61fa5f4f05fdb1795d360|a80ccd6c95a3aa08fcfb7a5a10ffc883344da48ec37ce605220f964cef14cb26
code sha256: 69f465fb3772305e1222a303541611e4daf5acf9c9cf75e8457dd376612b1ba1
git head: 95d5fe44b71d6f6fbd56e8fda9713b85983ea138
initial ranks: (1, 8, 12, 12, 12, 12, 1)
site-0 right orthogonality max error: 9.382572443428217e-07
test length, not scanned: 10000
rank-selection length, images not scanned: 5000
このセルは CUDA を初期化しません。


## 3. 1条件の実行

`train_scoped_full_batch_budget` が、scope に応じた全データ更新を行う。Notebook 側では測定用の更新式を再実装しない。

- `source`: 勾配を足す batch。DataLoader か、再走査できる GPU 常駐 tensor。
- `train_eval_loader`: 学習 loss と accuracy の評価。更新には使わない。
- `validation_loader`: early-stopping validation。best 選択に使う。
- `tt_learning_rate` と `dense_learning_rate`: 更新する側だけ数値。除外する側は `None`。

保存する history は、更新数 0 と 80 更新ごとの評価である。`is_best` はその時点の early-stopping validation loss が best なら真。同値の loss では、より早い checkpoint を残す。

local log の空欄は、その scope では計算していないという意味である。SVD の切り捨て量が 0 だったこととは区別する。

正常完了は、要求した更新数へ到達した `max_updates` だけである。非有限値は `non_finite`、CUDA のメモリ不足は `out_of_memory`。それ以外の例外は停止理由に変換しない。

1条件が終わることは、その scope が他の scope より良いことを保証しない。


In [35]:
def rate_text(value):
    """再開判定用の表記。除外は 0 ではなく excluded。"""
    if value is None:
        return "excluded"
    return value


def parse_rate(value):
    if value == "excluded":
        return None
    return float(value)


def parse_optional_int(value):
    if value == "excluded":
        return None
    return int(value)


def synchronize(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def require_cuda_device():
    from nn_compression.utils import set_seed

    if not torch.cuda.is_available():
        raise RuntimeError("この実験は GPU 常駐 batch を使うため、CUDA が必要です。")
    set_seed(SEED)
    # index 付きで返す。あとで tensor.device（cuda:0）と比較できるようにする。
    return torch.device("cuda", torch.cuda.current_device())


def preload_train(device):
    """前処理済みの学習 tensor を一度だけ device へ置く。"""
    synchronize(device)
    started = time.perf_counter()
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    images = train_images.to(device).contiguous()
    labels = train_labels.to(device).contiguous()
    synchronize(device)
    seconds = time.perf_counter() - started
    if images.shape != train_images.shape or labels.shape != train_labels.shape:
        raise RuntimeError("GPU 上の shape が CPU 基準と一致しません。")
    if images.dtype != train_images.dtype or labels.dtype != train_labels.dtype:
        raise RuntimeError("GPU 上の dtype が CPU 基準と一致しません。")
    if images.shape[0] != N_TRAIN:
        raise RuntimeError("GPU 上の sample 数が 50,000 ではありません。")
    if tensor_sha256(images) != IMAGE_SHA256 or tensor_sha256(labels) != LABEL_SHA256:
        raise RuntimeError("GPU 上の checksum が CPU 基準と一致しません。")
    for index in CHECK_INDICES:
        if not torch.equal(images[index].cpu(), train_images[index]):
            raise RuntimeError(f"GPU 上の境界画像 {index} が一致しません。")
        if int(labels[index].cpu()) != int(train_labels[index]):
            raise RuntimeError(f"GPU 上の境界ラベル {index} が一致しません。")
    source = GpuResidentTensorBatches(images, labels)
    seen = 0
    for batch_images, batch_labels in source.batches(BATCH_SIZE):
        seen += int(batch_images.shape[0])
        if batch_images.device != device or batch_labels.device != device:
            raise RuntimeError("常駐 batch が指定 device から離れました。")
    if seen != N_TRAIN:
        raise RuntimeError("GPU 常駐 batch が 50,000 件を過不足なく走査できません。")
    allocated = reserved = None
    if device.type == "cuda":
        allocated = int(torch.cuda.max_memory_allocated(device))
        reserved = int(torch.cuda.max_memory_reserved(device))
    return source, seconds, allocated, reserved


def move_model_and_check(model, device):
    model.to(device)
    # model.to(torch.device("cuda")) 後の tensor.device は cuda:0 になる。
    # torch.device("cuda")（index なし）と != 比較すると、実体は移っているのに落ちる。
    expected = torch.device(device)
    if expected.type == "cuda" and expected.index is None:
        expected = torch.device("cuda", torch.cuda.current_device())
    for name, value in model.state_dict().items():
        if value.dtype != torch.float32 or value.device != expected:
            raise RuntimeError(
                f"{name} が学習 device の float32 ではありません: "
                f"dtype={value.dtype}, device={value.device}, expected={expected}"
            )
    return model


def run_scope(
    scope_name,
    source,
    batch_size,
    tt_learning_rate,
    dense_learning_rate,
    total_updates,
    local_steps,
    max_sweeps,
    device,
):
    """Stage C から独立に1条件を学習する。前の条件の state は渡さない。"""
    model, _ortho = build_model_from_checkpoint()
    move_model_and_check(model, device)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    result = train_scoped_full_batch_budget(
        model,
        source,
        train_eval_loader,
        validation_loader,
        scope=TTUpdateScope(scope_name),
        layer_name=LAYER_NAME,
        batch_size=batch_size,
        total_updates=total_updates,
        evaluation_interval=EVALUATION_INTERVAL,
        rank_cap=RANK_CAP,
        tt_learning_rate=tt_learning_rate,
        dense_learning_rate=dense_learning_rate,
        diagnostic_images=diagnostic_images,
        diagnostic_labels=diagnostic_labels,
        device=device,
        local_steps_per_pair=local_steps,
        max_sweeps=max_sweeps,
        center=0,
    )
    synchronize(device)
    allocated = reserved = None
    if device.type == "cuda":
        allocated = int(torch.cuda.max_memory_allocated(device))
        reserved = int(torch.cuda.max_memory_reserved(device))
    return model, result, allocated, reserved


def best_accuracy(result):
    chosen = [row for row in result.history if row["is_best"]]
    if len(chosen) != 1:
        raise RuntimeError("best 行が1つではありません。")
    return float(chosen[0]["early_stopping_validation_accuracy"])


def atomic_json(path, payload):
    assert_output_path(path)
    temporary = Path(path).with_name(Path(path).name + ".tmp")
    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8", newline="\n")
    os.replace(temporary, path)


def condition_config(spec):
    return {
        **spec,
        "layer_name": LAYER_NAME,
        "out_modes": list(OUT_MODES),
        "in_modes": list(IN_MODES),
        "early_stopping_enabled": EARLY_STOPPING_ENABLED,
        "git_head": GIT_HEAD,
    }


def expected_round_trip(local_steps):
    visits = [(index, "right") for index in range(N_CORES - 1)]
    visits += [(index, "left") for index in range(N_CORES - 2, -1, -1)]
    rows = []
    center = 0
    for pair_index, direction in visits:
        for local_step in range(local_steps):
            center_after = pair_index + 1 if direction == "right" else pair_index
            rows.append((pair_index, direction, local_step, center, center_after))
            center = center_after
    if center != 0:
        raise RuntimeError("1往復後の正準中心が 0 に戻りません。")
    return rows


def assert_pair_log(result, local_steps):
    expected = expected_round_trip(local_steps)
    actual = [
        (
            int(row["pair_index"]),
            row["direction"],
            int(row["local_step"]),
            int(row["center_before"]),
            int(row["center_after"]),
        )
        for row in result.local_updates
    ]
    if actual != expected:
        raise RuntimeError("pair 順、方向、または正準中心の移動が1往復の定義と一致しません。")
    if any(int(row["retained_rank"]) > RANK_CAP for row in result.local_updates):
        raise RuntimeError("retained rank が上限を超えています。")


def collect_logits(model, device):
    was_training = model.training
    model.eval()
    outputs = []
    with torch.no_grad():
        for images, _labels in validation_loader:
            outputs.append(model(images.to(device)).cpu())
    model.train(was_training)
    logits = torch.cat(outputs)
    return logits, logits.argmax(dim=1)


def tensor_rmse(left, right):
    difference = left.detach().float().cpu() - right.detach().float().cpu()
    return float(torch.sqrt(torch.mean(difference.square())))


def tensor_max_abs(left, right):
    return float((left.detach().cpu() - right.detach().cpu()).abs().max())


print("1条件の関数を定義しました。このセルは学習も CUDA 初期化もしません。")


1条件の関数を定義しました。このセルは学習も CUDA 初期化もしません。


## 4. smoke

`RUN_SMOKE` が `False` のあいだは何も学習しない。

smoke 1 は、同じ Stage C から joint を 80 更新する。旧側は Notebook 03 と同じ `train_tt_linear_full_batch_sweeps` を、batch 512 の DataLoader、TT 学習率 0.3、通常 parameter 学習率 0.3、local step 8、左右 1 往復で呼ぶ。新側は `train_scoped_full_batch_budget` の joint を、batch 4,096 の GPU 常駐で呼ぶ。新しい scope 実装を旧方式とは呼ばない。

合格は、early-stopping validation loss の絶対差が $10^{-3}$ 未満、accuracy の絶対差が $0.001$ 未満、かつ両方とも 80 更新を完了して有限であること。accuracy は 0 から 1 の割合なので、$0.001$ は 0.1 percentage point である。

TT 再構成 weight、通常 parameter、logits、予測一致、rank 履歴、切り捨て量、実行時間、peak memory は診断値として保存する。SVD の符号や gauge で core の要素が違っても、それだけでは不合格にしない。基準を満たさない場合、閾値は緩めず、主実験へ進まない。

smoke 2 は新しい供給方法で joint、tt-only、dense-only を各 80 更新する。smoke の state は主実験へ渡さない。

- joint: TT core と通常 parameter が変わり、対象 pair 外の core は各局所更新のあいだ固定され、80 更新が完了する。
- tt-only: TT core は変わり、通常 parameter は初期 state と完全一致し、通常 parameter の更新候補は作らない。pair 順、方向、中心移動、rank 上限を確認する。
- dense-only: 全 TT core と再構成 weight は初期値と一致し、通常 parameter は変わる。SVD と core 書き戻しの回数は 0。80 更新は比較用の評価ブロックであり、往復とは呼ばない。

非有限値とメモリ不足だけを停止理由として記録する。実装例外は smoke 失敗へ変換して継続しない。

smoke の保存は、3,200 更新が妥当だという保証にはならない。


In [36]:
def smoke_identity():
    return {
        "checkpoint_sha256": SHARED_CHECKPOINT_SHA256,
        "image_sha256": IMAGE_SHA256,
        "label_sha256": LABEL_SHA256,
        "split_index_sha256": SPLIT_INDEX_SHA256,
        "code_sha256": CODE_SHA256,
        "seed": SEED,
        "rank_cap": RANK_CAP,
        "smoke_updates": SMOKE_UPDATES,
        "evaluation_interval": EVALUATION_INTERVAL,
        "joint_tt_learning_rate": JOINT_TT_LEARNING_RATE,
        "joint_dense_learning_rate": JOINT_DENSE_LEARNING_RATE,
        "old_batch_size": SMOKE_OLD_BATCH_SIZE,
        "old_data_residency": "dataloader",
        "new_batch_size": BATCH_SIZE,
        "new_data_residency": DATA_RESIDENCY,
    }


def save_smoke_run(name, model, result, spec):
    if not SAVE_RESULTS:
        return
    directory = smoke_models_dir / name
    assert_output_path(directory)
    save_scoped_full_batch_run(directory, model, result, condition_config(spec))


def finite_history(result):
    values = []
    for row in result.history:
        values.extend(
            (
                row["train_loss"],
                row["train_accuracy"],
                row["early_stopping_validation_loss"],
                row["early_stopping_validation_accuracy"],
            )
        )
    return all(math.isfinite(value) for value in values)


def require_completed(result, label):
    if result.stop_reason == "non_finite":
        raise RuntimeError(f"{label} は非有限値で停止しました。主実験は始めません。")
    if result.stop_reason == "out_of_memory":
        raise RuntimeError(f"{label} はメモリ不足で停止しました。主実験は始めません。")
    if result.stop_reason != "max_updates" or result.completed_updates != SMOKE_UPDATES:
        raise RuntimeError(f"{label} が 80 更新を完了していません: {result.stop_reason}")
    if not finite_history(result):
        raise RuntimeError(f"{label} の評価に非有限値があります。")


def validation_at(result, updates):
    rows = [row for row in result.history if row["cumulative_updates"] == updates]
    if len(rows) != 1:
        raise RuntimeError(f"更新数 {updates} の評価が1行ではありません。")
    row = rows[0]
    return float(row["early_stopping_validation_loss"]), float(row["early_stopping_validation_accuracy"])


def dense_tensors(model):
    return {
        name: parameter.detach().cpu().clone()
        for name, parameter in model.named_parameters()
        if not name.startswith(f"{LAYER_NAME}.cores.")
    }


def core_tensors(model):
    return {
        name: parameter.detach().cpu().clone()
        for name, parameter in model.named_parameters()
        if name.startswith(f"{LAYER_NAME}.cores.")
    }


def reconstructed_weight(model):
    # 学習 device 上で contract する。smoke の「core が変わっていない」判定は
    # cores_exact で行い、ここを CPU 強制にして遅くしない。
    return tt_matrix_to_dense(list(model.fc1.cores), OUT_MODES, IN_MODES).detach().cpu()


def old_joint_metrics(result):
    """Notebook 03 の結果を、05 の比較で使う loss と accuracy に読み替える。"""
    if result.stop_reason != "max_sweeps" or result.completed_sweeps != 1:
        raise RuntimeError(
            "旧 joint は Notebook 03 の学習経路が 1 往復を完了していません: "
            f"{result.stop_reason} sweeps={result.completed_sweeps}"
        )
    if result.successful_local_updates != SMOKE_UPDATES:
        raise RuntimeError(
            f"旧 joint の成功更新数が 80 ではありません: {result.successful_local_updates}"
        )
    matches = [
        row
        for row in result.history
        if int(row["cumulative_successful_updates"]) == SMOKE_UPDATES
    ]
    if len(matches) != 1:
        raise RuntimeError("旧 joint の 80 更新時点の評価が 1 行ではありません。")
    loss = float(matches[0]["validation_loss"])
    accuracy = float(matches[0]["validation_acc"])
    if not math.isfinite(loss) or not math.isfinite(accuracy):
        raise RuntimeError("旧 joint の validation が有限ではありません。")
    return loss, accuracy


def run_smoke():
    device = require_cuda_device()
    report = smoke_identity()
    started = time.perf_counter()
    # 旧側は Notebook 03 の学習関数。scope 実装へ置き換えない。
    old_model, _ortho = build_model_from_checkpoint()
    move_model_and_check(old_model, device)
    old_loader = make_loader(train_dataset, SMOKE_OLD_BATCH_SIZE)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    synchronize(device)
    old_started = time.perf_counter()
    old_result = train_tt_linear_full_batch_sweeps(
        old_model,
        old_loader,
        train_eval_loader,
        validation_loader,
        layer_name=LAYER_NAME,
        rank_cap=RANK_CAP,
        tt_pair_learning_rate=JOINT_TT_LEARNING_RATE,
        dense_parameter_learning_rate=JOINT_DENSE_LEARNING_RATE,
        max_sweeps=1,
        local_steps_per_pair=LOCAL_STEPS_PER_PAIR,
        early_stopping_patience=1,
        early_stopping_min_delta=0.0,
        early_stopping_enabled=False,
        diagnostic_images=diagnostic_images,
        diagnostic_labels=diagnostic_labels,
        device=device,
        center=0,
    )
    synchronize(device)
    old_seconds = time.perf_counter() - old_started
    old_allocated = old_reserved = None
    if device.type == "cuda":
        old_allocated = int(torch.cuda.max_memory_allocated(device))
        old_reserved = int(torch.cuda.max_memory_reserved(device))
    old_loss, old_accuracy = old_joint_metrics(old_result)
    if SAVE_RESULTS:
        old_directory = smoke_models_dir / "joint_dataloader_bs512"
        assert_output_path(old_directory)
        save_tt_linear_full_batch_sweep_run(
            old_directory,
            old_model,
            old_result,
            {
                "condition_id": "joint_dataloader_bs512",
                "layer_name": LAYER_NAME,
                "out_modes": list(OUT_MODES),
                "in_modes": list(IN_MODES),
                "implementation": "train_tt_linear_full_batch_sweeps",
                "batch_size": SMOKE_OLD_BATCH_SIZE,
                "data_residency": "dataloader",
                "tt_pair_learning_rate": JOINT_TT_LEARNING_RATE,
                "dense_parameter_learning_rate": JOINT_DENSE_LEARNING_RATE,
                "max_sweeps": 1,
                "local_steps_per_pair": LOCAL_STEPS_PER_PAIR,
                "early_stopping_enabled": False,
                "checkpoint_sha256": SHARED_CHECKPOINT_SHA256,
            },
        )
    resident, preload_seconds, preload_allocated, preload_reserved = preload_train(device)
    new_model, new_result, new_allocated, new_reserved = run_scope(
        "joint",
        resident,
        BATCH_SIZE,
        JOINT_TT_LEARNING_RATE,
        JOINT_DENSE_LEARNING_RATE,
        SMOKE_UPDATES,
        LOCAL_STEPS_PER_PAIR,
        1,
        device,
    )
    require_completed(new_result, "新 joint")
    save_smoke_run(
        "joint_gpu_resident_bs4096",
        new_model,
        new_result,
        {
            "condition_id": "joint_gpu_resident_bs4096",
            "scope": "joint",
            "batch_size": BATCH_SIZE,
            "data_residency": DATA_RESIDENCY,
        },
    )
    new_loss, new_accuracy = validation_at(new_result, SMOKE_UPDATES)
    old_logits, old_predictions = collect_logits(old_model, device)
    new_logits, new_predictions = collect_logits(new_model, device)
    disagreements = int((old_predictions != new_predictions).sum())
    loss_gap = abs(new_loss - old_loss)
    accuracy_gap = abs(new_accuracy - old_accuracy)
    joint_passed = (
        loss_gap < SMOKE_VALIDATION_LOSS_ABSOLUTE_TOLERANCE
        and accuracy_gap < SMOKE_VALIDATION_ACCURACY_ABSOLUTE_TOLERANCE
    )
    report.update(
        {
            "old_new_joint_passed": joint_passed,
            "validation_loss_gap": loss_gap,
            "validation_accuracy_gap": accuracy_gap,
            "old_validation_loss": old_loss,
            "new_validation_loss": new_loss,
            "old_validation_accuracy": old_accuracy,
            "new_validation_accuracy": new_accuracy,
            "logits_rmse": tensor_rmse(old_logits, new_logits),
            "logits_max_abs": tensor_max_abs(old_logits, new_logits),
            "prediction_agreement": float((old_predictions == new_predictions).float().mean()),
            "prediction_disagreements": disagreements,
            "reconstructed_weight_rmse": tensor_rmse(
                reconstructed_weight(old_model),
                reconstructed_weight(new_model),
            ),
            "dense_parameter_max_abs": max(
                tensor_max_abs(old_value, dense_tensors(new_model)[name])
                for name, old_value in dense_tensors(old_model).items()
            ),
            "old_retained_ranks": [int(row["retained_rank"]) for row in old_result.local_updates],
            "new_retained_ranks": [int(row["retained_rank"]) for row in new_result.local_updates],
            "old_discarded_squared_norm": [
                float(row["discarded_squared_norm"]) for row in old_result.local_updates
            ],
            "new_discarded_squared_norm": [
                float(row["discarded_squared_norm"]) for row in new_result.local_updates
            ],
            "old_seconds_train": old_seconds,
            "old_implementation": "train_tt_linear_full_batch_sweeps",
            "new_implementation": "train_scoped_full_batch_budget",
            "new_seconds_train": new_result.seconds_train,
            "preload_seconds": preload_seconds,
            "old_peak_allocated_bytes": old_allocated,
            "old_peak_reserved_bytes": old_reserved,
            "new_peak_allocated_bytes": new_allocated,
            "new_peak_reserved_bytes": new_reserved,
            "preload_peak_allocated_bytes": preload_allocated,
            "preload_peak_reserved_bytes": preload_reserved,
        }
    )
    if SAVE_RESULTS:
        atomic_json(smoke_results_dir / "smoke_report.json", report)
    if not joint_passed:
        raise RuntimeError(
            "smoke 1 が validation の基準を満たしません。"
            "閾値は緩めず、主実験は始めません。"
            f" loss差={loss_gap}, accuracy差={accuracy_gap}"
        )

    scope_results = {}
    for scope_name, tt_rate, dense_rate, local_steps, sweeps in (
        ("joint", JOINT_TT_LEARNING_RATE, JOINT_DENSE_LEARNING_RATE, LOCAL_STEPS_PER_PAIR, 1),
        ("tt_only", TT_ONLY_TT_LEARNING_RATE, TT_ONLY_DENSE_LEARNING_RATE, LOCAL_STEPS_PER_PAIR, 1),
        ("dense_only", DENSE_ONLY_TT_LEARNING_RATE, DENSE_ONLY_DENSE_LEARNING_RATE, None, None),
    ):
        fresh, _ortho = build_model_from_checkpoint()
        initial_dense = dense_tensors(fresh)
        initial_cores = core_tensors(fresh)
        del fresh
        model, result, _allocated, _reserved = run_scope(
            scope_name,
            resident,
            BATCH_SIZE,
            tt_rate,
            dense_rate,
            SMOKE_UPDATES,
            local_steps,
            sweeps,
            device,
        )
        require_completed(result, scope_name)
        save_smoke_run(
            f"scope_{scope_name}",
            model,
            result,
            {"condition_id": f"scope_{scope_name}", "scope": scope_name, "batch_size": BATCH_SIZE},
        )
        changed_cores = any(
            not torch.equal(initial_cores[name], value) for name, value in core_tensors(model).items()
        )
        changed_dense = any(
            not torch.equal(initial_dense[name], value) for name, value in dense_tensors(model).items()
        )
        dense_exact = all(
            torch.equal(initial_dense[name], value) for name, value in dense_tensors(model).items()
        )
        cores_exact = all(
            torch.equal(initial_cores[name], value) for name, value in core_tensors(model).items()
        )
        if scope_name in {"joint", "tt_only"}:
            assert_pair_log(result, LOCAL_STEPS_PER_PAIR)
        if scope_name == "joint":
            checks = {
                "changed_cores": changed_cores,
                "changed_dense": changed_dense,
                "dense_candidate_count": result.dense_candidate_count == SMOKE_UPDATES,
            }
        elif scope_name == "tt_only":
            checks = {
                "changed_cores": changed_cores,
                "dense_exact": dense_exact,
                "dense_candidate_count": result.dense_candidate_count == 0,
                "svd_call_count": result.svd_call_count == SMOKE_UPDATES,
            }
        else:
            # weight_exact は使わない。初期は CPU・学習後は GPU で contract すると
            # 同じ core でもビット一致しない。TT が凍結かの本判定は cores_exact。
            checks = {
                "cores_exact": cores_exact,
                "changed_dense": changed_dense,
                "svd_call_count": result.svd_call_count == 0,
                "core_writeback_count": result.core_writeback_count == 0,
                "center_is_none": result.center is None,
                "pair_index_none": all(row["pair_index"] is None for row in result.local_updates),
                "discarded_none": all(
                    row["discarded_squared_norm"] is None for row in result.local_updates
                ),
                "tt_gradient_none": all(
                    row["tt_gradient_norm"] is None for row in result.local_updates
                ),
            }
        passed = all(checks.values())
        scope_results[scope_name] = passed
        report[f"{scope_name}_scope_passed"] = passed
        report[f"{scope_name}_scope_checks"] = checks
        if not passed:
            report["scope_checks_passed"] = False
            if SAVE_RESULTS:
                atomic_json(smoke_results_dir / "smoke_report.json", report)
            failed = [name for name, ok in checks.items() if not ok]
            raise RuntimeError(
                f"smoke 2 の {scope_name} が更新対象の検証を満たしません。"
                f"失敗項目: {failed}。主実験は始めません。"
            )
    report["scope_checks_passed"] = all(scope_results.values())
    report["smoke_seconds_total"] = time.perf_counter() - started
    if SAVE_RESULTS:
        atomic_json(smoke_results_dir / "smoke_report.json", report)
    print("smoke 1 と smoke 2 は基準を満たしました。主実験はこの記録を確認してから始めてください。")
    return report


if not RUN_SMOKE:
    print("RUN_SMOKE は False です。80 更新の smoke は実行しません。")
else:
    smoke_report = run_smoke()
    print("old/new joint passed:", smoke_report["old_new_joint_passed"])
    print("scope checks passed:", smoke_report["scope_checks_passed"])


smoke 1 と smoke 2 は基準を満たしました。主実験はこの記録を確認してから始めてください。
old/new joint passed: True
scope checks passed: True


## 5. 主実験

`RUN_MAIN_EXPERIMENT` が `False` のあいだは学習しない。

開始前に、保存済み smoke の checkpoint SHA-256、データ checksum、分割識別子、code SHA-256、設定、旧方式と新方式の joint 合格、3 scope の合格が、いまの設定と一致することを要求する。不一致や未完了では理由を表示して停止する。閾値が足りない記録を、閾値を緩めて通すことはしない。

各方式は Stage C から作り直す。前の方式の最終モデルは渡さない。joint と tt-only は 40 往復、合計 3,200 更新である。dense-only は 3,200 回の通常 parameter 更新で、80 更新は評価ブロックであり往復ではない。

best は early-stopping validation loss の最小で、初期状態も含む。loss が同値なら累積更新数が小さい方を選ぶ。best accuracy はその行の accuracy である。final は 3,200 更新時点で、best とは別に保存する。

再開で skip できるのは、`stop_reason` が `max_updates` かつ成功更新数が 3,200 の条件だけである。`non_finite`、`out_of_memory`、更新数不足は診断用の成果物だけを残し、集計行は追加しない。途中ディレクトリは削除も上書きもしない。

主実験の完了は、その予算での到達値を示す。同じ実時間での比較でも、収束後の最適値でもない。


In [37]:
def main_spec(scope_name):
    if scope_name == "joint":
        tt_rate, dense_rate, local_steps, sweeps = (
            JOINT_TT_LEARNING_RATE,
            JOINT_DENSE_LEARNING_RATE,
            LOCAL_STEPS_PER_PAIR,
            MAX_SWEEPS,
        )
    elif scope_name == "tt_only":
        tt_rate, dense_rate, local_steps, sweeps = (
            TT_ONLY_TT_LEARNING_RATE,
            TT_ONLY_DENSE_LEARNING_RATE,
            LOCAL_STEPS_PER_PAIR,
            MAX_SWEEPS,
        )
    elif scope_name == "dense_only":
        tt_rate, dense_rate, local_steps, sweeps = (
            DENSE_ONLY_TT_LEARNING_RATE,
            DENSE_ONLY_DENSE_LEARNING_RATE,
            None,
            None,
        )
    else:
        raise ValueError(scope_name)
    return {
        "condition_id": scope_name,
        "scope": scope_name,
        "batch_size": BATCH_SIZE,
        "data_residency": DATA_RESIDENCY,
        "tt_learning_rate": rate_text(tt_rate),
        "dense_learning_rate": rate_text(dense_rate),
        "total_updates": TOTAL_FULL_BATCH_UPDATES,
        "evaluation_interval": EVALUATION_INTERVAL,
        "local_steps_per_pair": rate_text(local_steps),
        "max_sweeps": rate_text(sweeps),
        "seed": SEED,
        "rank_cap": RANK_CAP,
        "checkpoint_sha256": SHARED_CHECKPOINT_SHA256,
        "code_sha256": CODE_SHA256,
        "split_index_sha256": SPLIT_INDEX_SHA256,
        "image_sha256": IMAGE_SHA256,
        "label_sha256": LABEL_SHA256,
    }


def load_smoke_report():
    path = smoke_results_dir / "smoke_report.json"
    if not path.is_file():
        raise RuntimeError("smoke の記録がありません。主実験は開始しません。")
    return json.loads(path.read_text(encoding="utf-8"))


def run_main_condition(spec, source, preload_seconds, preload_allocated, preload_reserved):
    # preload 済みの常駐 batch と同じ device を使う。
    # torch.device("cuda")（index なし）だと source.device（cuda:0）と不一致になる。
    device = source.device
    started_save = None
    model, result, allocated, reserved = run_scope(
        spec["scope"],
        source,
        BATCH_SIZE,
        parse_rate(spec["tt_learning_rate"]),
        parse_rate(spec["dense_learning_rate"]),
        TOTAL_FULL_BATCH_UPDATES,
        parse_optional_int(spec["local_steps_per_pair"]),
        parse_optional_int(spec["max_sweeps"]),
        device,
    )
    directory = models_dir / spec["condition_id"]
    assert_output_path(directory)
    synchronize(device)
    started_save = time.perf_counter()
    if SAVE_RESULTS:
        save_scoped_full_batch_run(directory, model, result, condition_config(spec))
    seconds_save = time.perf_counter() - started_save
    initial_loss = float(result.history[0]["early_stopping_validation_loss"])
    initial_accuracy = float(result.history[0]["early_stopping_validation_accuracy"])
    final_row = result.history[-1]
    row = dict(spec)
    row.update(
        {
            "stop_reason": result.stop_reason,
            "completed_updates": result.completed_updates,
            "best_update_count": result.best_update_count,
            "best_validation_loss": result.best_validation_loss,
            "best_validation_accuracy": best_accuracy(result),
            "final_validation_loss": float(final_row["early_stopping_validation_loss"]),
            "final_validation_accuracy": float(final_row["early_stopping_validation_accuracy"]),
            "initial_validation_loss": initial_loss,
            "initial_validation_accuracy": initial_accuracy,
            "preload_seconds": preload_seconds,
            "preload_peak_allocated_bytes": preload_allocated,
            "preload_peak_reserved_bytes": preload_reserved,
            "seconds_train": result.seconds_train,
            "seconds_eval": result.seconds_eval,
            "seconds_save": seconds_save,
            "seconds_total": result.seconds_train + result.seconds_eval + seconds_save,
            "peak_allocated_bytes": allocated,
            "peak_reserved_bytes": reserved,
            "svd_call_count": result.svd_call_count,
            "dense_candidate_count": result.dense_candidate_count,
            "core_writeback_count": result.core_writeback_count,
        }
    )
    # 診断用の成果物は残す。集計行は正常完了のときだけ返す。
    assert_scope_budget_completed(
        result.stop_reason,
        result.completed_updates,
        TOTAL_FULL_BATCH_UPDATES,
    )
    return row


if not RUN_MAIN_EXPERIMENT:
    print("RUN_MAIN_EXPERIMENT は False です。3,200 更新の主実験は実行しません。")
else:
    # 再開可能な集計は、条件ディレクトリの成果物がある前提。
    # SAVE_RESULTS=False だと学習後に成果物検査で落ちる。
    if not SAVE_RESULTS:
        raise RuntimeError(
            "主実験は SAVE_RESULTS=True が必要です。"
            "False のままだとモデルを保存せず、集計CSVへ書けません。"
        )
    expected_smoke = smoke_identity()
    expected_smoke.update(
        {
            "old_new_joint_passed": True,
            "joint_scope_passed": True,
            "tt_only_scope_passed": True,
            "dense_only_scope_passed": True,
            "scope_checks_passed": True,
        }
    )
    try:
        require_recorded_smoke_gate(load_smoke_report(), expected_smoke, SMOKE_GATE_FIELDS)
    except RuntimeError as exc:
        print(str(exc))
        raise
    summary_path = results_dir / "condition_summary.csv"
    assert_output_path(summary_path)
    planned = [main_spec(name) for name in ("joint", "tt_only", "dense_only")]
    reject_incomplete_scope_rows(
        load_condition_summary(summary_path),
        ("joint", "tt_only", "dense_only"),
        TOTAL_FULL_BATCH_UPDATES,
    )
    prepared = {}

    def run_one(spec):
        # 未実行条件があるときだけ GPU へ配置する。完了済みの skip では CUDA を使わない。
        if "source" not in prepared:
            device = require_cuda_device()
            source, seconds, allocated, reserved = preload_train(device)
            prepared["source"] = source
            prepared["seconds"] = seconds
            prepared["allocated"] = allocated
            prepared["reserved"] = reserved
        return run_main_condition(
            spec,
            prepared["source"],
            prepared["seconds"],
            prepared["allocated"],
            prepared["reserved"],
        )

    condition_summary = run_resumable_condition_grid(
        planned,
        summary_path=summary_path,
        models_dir=models_dir,
        fieldnames=SUMMARY_FIELDS,
        setting_fields=SETTING_FIELDS,
        run_one=run_one,
    )
    print("主実験の記録行数:", len(condition_summary))
    print("preload_seconds は3条件で共有する一度だけの配置時間であり、seconds_total には含めません。")


主実験の記録行数: 3
preload_seconds は3条件で共有する一度だけの配置時間であり、seconds_total には含めません。


## 6. 比較と解釈

主比較の横軸は累積 full-batch 更新数である。補助として、学習と評価に使った累積時間も描く。GPU 事前配置の時間は、この時間軸に足さない。

改善量を正の値で読むときは、次の符号にする。

$$
\Delta \mathrm{loss} = L_{\mathrm{ref}} - L_{\mathrm{candidate}}
$$

$$
\Delta \mathrm{accuracy} = A_{\mathrm{candidate}} - A_{\mathrm{ref}}
$$

Notebook 03 の学習率 0.3 は参考値である。batch 512 の DataLoader で 3,200 更新した validation loss は 0.3426136145591736、accuracy は 0.8710。これは旧実装と新実装の長期的な参考であり、同じ実装の反復でも、採用条件の主比較でもない。

この結果だけでは、次のことは言えない。

- 同じ実時間での比較ではない。
- 各方式が収束したあとの最適性能ではない。
- 学習率 0.3 を方式ごとに選び直していない。
- seed と初期 checkpoint は一つである。
- joint と dense-only の差には、学習経路の相互作用も含まれる。
- DMRG に似た更新が、他の TT optimizer より優れることは示さない。
- 最大更新数の時点でまだ改善しているなら、到達性能は未確定である。

比較図と `initial_final_best.csv` は、joint、tt-only、dense-only の3条件がすべて正常完了し、記録された設定が現在の設定と一致し、最終更新数が 3,200 のときだけ書く。1条件または2条件だけのときは未完了を表示して SKIP し、既存の成果物は変更しない。集計行があるのに設定や更新数が不正なときは SKIP せずエラーにする。


In [38]:
def same_setting(recorded, expected):
    if recorded == expected or str(recorded) == str(expected):
        return True
    try:
        return math.isclose(float(recorded), float(expected), rel_tol=1e-9, abs_tol=0.0)
    except (TypeError, ValueError):
        return False


def scope_observations(summary_rows):
    by_id = {}
    for row in summary_rows:
        condition_id = row.get("condition_id")
        if condition_id in by_id:
            raise ValueError(f"集計の condition_id が重複しています: {condition_id}")
        by_id[condition_id] = row
    observations = {}
    for scope_name in ("joint", "tt_only", "dense_only"):
        row = by_id.get(scope_name)
        if row is None:
            observations[scope_name] = None
            continue
        history_path = models_dir / scope_name / "history.csv"
        final_updates = None
        if history_path.is_file():
            with history_path.open(encoding="utf-8", newline="") as handle:
                history_rows = list(csv.DictReader(handle))
            if history_rows:
                final_updates = int(float(history_rows[-1]["cumulative_updates"]))
        expected = main_spec(scope_name)
        observations[scope_name] = {
            "settings_match": all(
                same_setting(row.get(field), expected[field]) for field in SETTING_FIELDS
            ),
            "history_exists": history_path.is_file() and final_updates is not None,
            "stop_reason": row.get("stop_reason"),
            "completed_updates": int(float(row["completed_updates"])),
            "final_updates": final_updates,
        }
    return observations


summary_path = results_dir / "condition_summary.csv"
comparison_ready = False
if not summary_path.is_file():
    print("SKIP: 主実験の condition_summary.csv はまだありません。比較図は作りません。")
else:
    import csv

    with summary_path.open(encoding="utf-8", newline="") as handle:
        summary_rows = list(csv.DictReader(handle))
    missing_scopes = incomplete_scope_names(
        scope_observations(summary_rows),
        ("joint", "tt_only", "dense_only"),
        TOTAL_FULL_BATCH_UPDATES,
    )
    if missing_scopes:
        print("SKIP: 比較には3条件の正常完了が必要です。未完了:", ", ".join(missing_scopes))
    else:
        comparison_ready = True

if comparison_ready:
    import matplotlib.pyplot as plt
    import pandas as pd

    plt.rcParams["font.family"] = ["Yu Gothic", "Meiryo", "MS Gothic", "Noto Sans CJK JP", "DejaVu Sans"]
    histories = {
        scope_name: pd.read_csv(models_dir / scope_name / "history.csv")
        for scope_name in ("joint", "tt_only", "dense_only")
    }

    def save_figure(figure, name):
        if not SAVE_RESULTS:
            plt.show()
            return
        path = results_dir / name
        assert_output_path(path)
        temporary = path.with_name(path.name + ".tmp.png")
        figure.savefig(temporary, dpi=120, bbox_inches="tight")
        os.replace(temporary, path)
        plt.close(figure)

    def plot_curves(x_column, x_label, y_column, y_label, filename):
        figure, axis = plt.subplots(figsize=(7.5, 4.5))
        for scope_name, frame in histories.items():
            axis.plot(frame[x_column], frame[y_column], marker="o", label=scope_name)
        axis.set_xlabel(x_label)
        axis.set_ylabel(y_label)
        axis.legend()
        figure.tight_layout()
        save_figure(figure, filename)

    plot_curves(
        "cumulative_updates",
        "累積 full-batch 更新数",
        "early_stopping_validation_loss",
        "early-stopping validation loss",
        "validation_loss_vs_updates.png",
    )
    plot_curves(
        "cumulative_updates",
        "累積 full-batch 更新数",
        "early_stopping_validation_accuracy",
        "early-stopping validation accuracy",
        "validation_accuracy_vs_updates.png",
    )
    for frame in histories.values():
        frame["measured_seconds"] = frame["cumulative_train_seconds"] + frame["cumulative_eval_seconds"]
    plot_curves(
        "measured_seconds",
        "累積の学習時間と評価時間（秒、GPU事前配置を除く）",
        "early_stopping_validation_loss",
        "early-stopping validation loss",
        "validation_loss_vs_time.png",
    )
    plot_curves(
        "measured_seconds",
        "累積の学習時間と評価時間（秒、GPU事前配置を除く）",
        "early_stopping_validation_accuracy",
        "early-stopping validation accuracy",
        "validation_accuracy_vs_time.png",
    )

    comparison_rows = []
    for scope_name, frame in histories.items():
        initial = frame.iloc[0]
        final = frame.iloc[-1]
        best = frame.loc[frame["is_best"].astype(str).str.lower().eq("true")].iloc[0]
        comparison_rows.append(
            {
                "scope": scope_name,
                "initial_loss": float(initial["early_stopping_validation_loss"]),
                "best_loss": float(best["early_stopping_validation_loss"]),
                "final_loss": float(final["early_stopping_validation_loss"]),
                "initial_accuracy": float(initial["early_stopping_validation_accuracy"]),
                "best_accuracy": float(best["early_stopping_validation_accuracy"]),
                "final_accuracy": float(final["early_stopping_validation_accuracy"]),
                "best_updates": int(best["cumulative_updates"]),
                "final_updates": int(final["cumulative_updates"]),
                "loss_improvement_from_initial": float(initial["early_stopping_validation_loss"])
                - float(final["early_stopping_validation_loss"]),
                "accuracy_improvement_from_initial": float(final["early_stopping_validation_accuracy"])
                - float(initial["early_stopping_validation_accuracy"]),
            }
        )
        if len(frame) >= 2 and float(final["early_stopping_validation_loss"]) < float(
            frame.iloc[-2]["early_stopping_validation_loss"]
        ):
            print(f"{scope_name}: 最終区間でも validation loss が下がっています。到達性能は未確定です。")
    comparison = pd.DataFrame(comparison_rows)
    if SAVE_RESULTS:
        target = results_dir / "initial_final_best.csv"
        assert_output_path(target)
        temporary = target.with_name(target.name + ".tmp")
        comparison.to_csv(temporary, index=False)
        os.replace(temporary, target)
    print(comparison.to_string(index=False))

    joint = comparison.loc[comparison["scope"].eq("joint")].iloc[0]
    reference_loss = NOTEBOOK_03_REFERENCE["validation_loss"]
    reference_accuracy = NOTEBOOK_03_REFERENCE["validation_accuracy"]
    print("Notebook 03 学習率 0.3 は参考比較です。主比較ではありません。")
    print("参考 loss:", reference_loss, "参考 accuracy:", reference_accuracy)
    print(
        "05 joint final との参考差 loss:",
        reference_loss - float(joint["final_loss"]),
        "accuracy:",
        float(joint["final_accuracy"]) - reference_accuracy,
    )
    print("batch、供給方法、実装が違うため、この差を同一条件の反復実験としては読みません。")


findfont: Font family 'Noto Sans CJK JP' not found.
findfont: Font family 'Noto Sans CJK JP' not found.
findfont: Font family 'Noto Sans CJK JP' not found.
findfont: Font family 'Noto Sans CJK JP' not found.
findfont: Font family 'Noto Sans CJK JP' not found.
findfont: Font family 'Noto Sans CJK JP' not found.
findfont: Font family 'Noto Sans CJK JP' not found.
findfont: Font family 'Noto Sans CJK JP' not found.
findfont: Font family 'Noto Sans CJK JP' not found.
findfont: Font family 'Noto Sans CJK JP' not found.
findfont: Font family 'Noto Sans CJK JP' not found.
findfont: Font family 'Noto Sans CJK JP' not found.
findfont: Font family 'Noto Sans CJK JP' not found.
findfont: Font family 'Noto Sans CJK JP' not found.
findfont: Font family 'Noto Sans CJK JP' not found.
findfont: Font family 'Noto Sans CJK JP' not found.
findfont: Font family 'Noto Sans CJK JP' not found.
findfont: Font family 'Noto Sans CJK JP' not found.
findfont: Font family 'Noto Sans CJK JP' not found.
findfont: Fo

joint: 最終区間でも validation loss が下がっています。到達性能は未確定です。
tt_only: 最終区間でも validation loss が下がっています。到達性能は未確定です。
     scope  initial_loss  best_loss  final_loss  initial_accuracy  best_accuracy  final_accuracy  best_updates  final_updates  loss_improvement_from_initial  accuracy_improvement_from_initial
     joint      2.303983   0.341941    0.341941            0.0946         0.8716          0.8716          3200           3200                       1.962042                             0.7770
   tt_only      2.303983   1.236367    1.236367            0.0946         0.6278          0.6278          3200           3200                       1.067616                             0.5332
dense_only      2.303983   1.044149    1.890252            0.0946         0.6660          0.3076          1360           3200                       0.413730                             0.2130
Notebook 03 学習率 0.3 は参考比較です。主比較ではありません。
参考 loss: 0.3426136145591736 参考 accuracy: 0.871
05 joint final との参考差 loss: 0.000672735548